# Forecasting - Inferencia

Notebook para generar el forecasting de ventas a partir de los datos de inferencia.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [4]:
df_inferencia = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
df_inferencia.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Preparación de la estructura objetivo

Se toma como referencia la estructura de columnas de `df_integrado.csv` generado en el notebook de entrenamiento, para garantizar que `df_inferencia_transformado` quede con exactamente las mismas variables.

In [5]:
ruta_estructura = '../data/processed/df_integrado.csv'
columnas_objetivo = pd.read_csv(ruta_estructura, nrows=0).columns.tolist()

df_inferencia_transformado = df_inferencia.copy()
df_inferencia_transformado['fecha'] = pd.to_datetime(df_inferencia_transformado['fecha'], errors='coerce')

if df_inferencia_transformado['fecha'].isna().any():
    raise ValueError('La columna fecha contiene valores no válidos después de convertirla a datetime.')

print(f'Columnas objetivo (según df_integrado): {len(columnas_objetivo)}')
print(f'Filas y columnas de partida: {df_inferencia_transformado.shape}')

Columnas objetivo (según df_integrado): 97
Filas y columnas de partida: (888, 13)


## Creación de variables temporales y de calendario

Se replica exactamente la misma lógica aplicada sobre `df_integrado` en el notebook de entrenamiento: componentes de calendario, festivos de México, Black Friday / Cyber Monday y codificación cíclica.

In [6]:
fecha = df_inferencia_transformado['fecha']
anios_calendario = sorted(fecha.dt.year.unique().tolist())
festivos_mx = holidays.MX(years=anios_calendario, language='es')
festivos_mx = {pd.Timestamp(fecha_festivo): nombre for fecha_festivo, nombre in festivos_mx.items()}

# Fechas de campañas comerciales: último viernes de noviembre y lunes siguiente.
def ultimo_viernes_noviembre(anio):
    ultimo_dia = pd.Timestamp(int(anio), 11, 30)
    return ultimo_dia - pd.Timedelta(days=(ultimo_dia.weekday() - 4) % 7)

black_fridays_calendario = {
    int(anio): ultimo_viernes_noviembre(anio)
    for anio in anios_calendario
}
cyber_mondays_calendario = {
    anio: fecha_bf + pd.Timedelta(days=3)
    for anio, fecha_bf in black_fridays_calendario.items()
}

fecha_solo_dia = fecha.dt.normalize()
black_friday_serie = fecha_solo_dia.map(
    {fecha_bf: True for fecha_bf in black_fridays_calendario.values()}
).fillna(False).astype(bool)
cyber_monday_serie = fecha_solo_dia.map(
    {fecha_cm: True for fecha_cm in cyber_mondays_calendario.values()}
).fillna(False).astype(bool)

# Componentes básicos del calendario.
df_inferencia_transformado['anio'] = fecha.dt.year.astype('int16')
df_inferencia_transformado['mes'] = fecha.dt.month.astype('int8')
df_inferencia_transformado['trimestre'] = fecha.dt.quarter.astype('int8')
df_inferencia_transformado['dia_mes'] = fecha.dt.day.astype('int8')
df_inferencia_transformado['dia_anio'] = fecha.dt.dayofyear.astype('int16')
df_inferencia_transformado['dia_semana'] = fecha.dt.dayofweek.astype('int8')
nombres_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
nombres_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
df_inferencia_transformado['nombre_dia'] = fecha.dt.dayofweek.map(dict(enumerate(nombres_dias)))
df_inferencia_transformado['nombre_mes'] = fecha.dt.month.map(dict(enumerate(nombres_meses, start=1)))
df_inferencia_transformado['semana_iso'] = fecha.dt.isocalendar().week.astype('int8')
df_inferencia_transformado['semana_mes'] = ((fecha.dt.day - 1) // 7 + 1).astype('int8')

# Indicadores de periodo.
df_inferencia_transformado['es_fin_semana'] = (df_inferencia_transformado['dia_semana'] >= 5)
df_inferencia_transformado['es_inicio_mes'] = fecha.dt.is_month_start
df_inferencia_transformado['es_fin_mes'] = fecha.dt.is_month_end
df_inferencia_transformado['es_inicio_trimestre'] = fecha.dt.is_quarter_start
df_inferencia_transformado['es_fin_trimestre'] = fecha.dt.is_quarter_end
df_inferencia_transformado['es_inicio_anio'] = fecha.dt.is_year_start
df_inferencia_transformado['es_fin_anio'] = fecha.dt.is_year_end
df_inferencia_transformado['es_quincena'] = fecha.dt.day.isin([1, 15])

# Festivos oficiales de México y campañas comerciales.
festivos_mx_por_dia = {fecha_festivo.date(): nombre for fecha_festivo, nombre in festivos_mx.items()}
df_inferencia_transformado['es_festivo'] = fecha.dt.date.isin(festivos_mx_por_dia).astype(bool)
df_inferencia_transformado['nombre_festivo'] = fecha.dt.date.map(festivos_mx_por_dia).fillna('No festivo')
df_inferencia_transformado['es_black_friday'] = black_friday_serie
df_inferencia_transformado['es_cyber_monday'] = cyber_monday_serie
# Distancia en días respecto al Black Friday del mismo año.
fecha_bf_por_fila = fecha.dt.year.map(black_fridays_calendario)
df_inferencia_transformado['dias_desde_black_friday'] = (
    fecha_solo_dia - pd.to_datetime(fecha_bf_por_fila).dt.normalize()
).dt.days.astype('int16')
df_inferencia_transformado['dias_hasta_black_friday'] = (-df_inferencia_transformado['dias_desde_black_friday']).astype('int16')

# Distancias y codificación cíclica para capturar estacionalidad.
df_inferencia_transformado['dias_desde_inicio_datos'] = (fecha_solo_dia - fecha_solo_dia.min()).dt.days.astype('int32')
df_inferencia_transformado['dias_hasta_fin_mes'] = (fecha.dt.days_in_month - fecha.dt.day).astype('int8')
df_inferencia_transformado['dia_semana_sin'] = np.sin(2 * np.pi * df_inferencia_transformado['dia_semana'] / 7)
df_inferencia_transformado['dia_semana_cos'] = np.cos(2 * np.pi * df_inferencia_transformado['dia_semana'] / 7)
df_inferencia_transformado['mes_sin'] = np.sin(2 * np.pi * (df_inferencia_transformado['mes'] - 1) / 12)
df_inferencia_transformado['mes_cos'] = np.cos(2 * np.pi * (df_inferencia_transformado['mes'] - 1) / 12)
df_inferencia_transformado['dia_anio_sin'] = np.sin(2 * np.pi * (df_inferencia_transformado['dia_anio'] - 1) / 365.25)
df_inferencia_transformado['dia_anio_cos'] = np.cos(2 * np.pi * (df_inferencia_transformado['dia_anio'] - 1) / 365.25)

print(f'Variables temporales creadas. Dimensiones actuales: {df_inferencia_transformado.shape}')
print(f'Festivos mexicanos en calendario: {len(festivos_mx)} fechas')
print(f'Black Fridays identificados: {df_inferencia_transformado["es_black_friday"].sum()} registros')
print(f'Cyber Mondays identificados: {df_inferencia_transformado["es_cyber_monday"].sum()} registros')

Variables temporales creadas. Dimensiones actuales: (888, 45)
Festivos mexicanos en calendario: 7 fechas
Black Fridays identificados: 24 registros
Cyber Mondays identificados: 0 registros


## Variables de lags y media móvil

Se calculan los lags de `unidades_vendidas` (1 a 7 días) y la media móvil de 7 días por `producto_id`, agrupando también por año para replicar el mismo criterio del entrenamiento.

In [7]:
df_inferencia_transformado = df_inferencia_transformado.sort_values(['producto_id', 'fecha']).copy()
df_inferencia_transformado['anio_lag'] = df_inferencia_transformado['fecha'].dt.year
grupo_lags = df_inferencia_transformado.groupby(['producto_id', 'anio_lag'], sort=False)['unidades_vendidas']
columnas_lags = []
for periodo_lag in range(1, 8):
    nombre_lag = f'lag_{periodo_lag}_unidades_vendidas'
    df_inferencia_transformado[nombre_lag] = grupo_lags.shift(periodo_lag)
    columnas_lags.append(nombre_lag)

df_inferencia_transformado['media_movil_7_dias'] = grupo_lags.transform(
    lambda serie: serie.shift(1).rolling(window=7, min_periods=7).mean()
)
columnas_nuevas_lags = columnas_lags + ['media_movil_7_dias']
df_inferencia_transformado = df_inferencia_transformado.drop(columns='anio_lag')

# IMPORTANTE: en inferencia, 'unidades_vendidas' solo es real para octubre
# (los datos históricos conocidos); en noviembre es NaN porque es justamente
# lo que se quiere predecir. Por eso, a diferencia del entrenamiento, NO se
# eliminan aquí las filas con lags/media_movil en NaN: hacerlo borraría de
# forma prematura casi todos los registros de noviembre. El modelo
# (HistGradientBoostingRegressor) admite valores NaN de forma nativa, así
# que estas columnas se dejan tal cual y solo se filtra el mes al final.
print(f'Filas tras crear lags (sin eliminar nulos): {len(df_inferencia_transformado)}')
print(f'Nulos en columnas de lags (esperado en noviembre): {df_inferencia_transformado[columnas_nuevas_lags].isna().sum().to_dict()}')
print(f'Variables creadas: {columnas_nuevas_lags}')

Filas tras crear lags (sin eliminar nulos): 888
Nulos en columnas de lags (esperado en noviembre): {'lag_1_unidades_vendidas': 720, 'lag_2_unidades_vendidas': 720, 'lag_3_unidades_vendidas': 720, 'lag_4_unidades_vendidas': 720, 'lag_5_unidades_vendidas': 720, 'lag_6_unidades_vendidas': 720, 'lag_7_unidades_vendidas': 720, 'media_movil_7_dias': 864}
Variables creadas: ['lag_1_unidades_vendidas', 'lag_2_unidades_vendidas', 'lag_3_unidades_vendidas', 'lag_4_unidades_vendidas', 'lag_5_unidades_vendidas', 'lag_6_unidades_vendidas', 'lag_7_unidades_vendidas', 'media_movil_7_dias']


## Variables de precio y competencia

Se calcula el porcentaje de descuento sobre el precio base y se combina la información de `df_competencia` en `precio_Competencia` y `ratio_precio`, igual que en el notebook de entrenamiento.

In [8]:
df_inferencia_transformado['descuento_porcentaje'] = (
    (df_inferencia_transformado['precio_venta'] - df_inferencia_transformado['precio_base'])
    / df_inferencia_transformado['precio_base']
) * 100

columnas_competencia = ['Amazon', 'Decathlon', 'Deporvillage']
# La celda es idempotente: si ya se ejecutó antes y las columnas de
# competencia fueron eliminadas, no se recalculan precio_Competencia/ratio_precio.
if all(columna in df_inferencia_transformado.columns for columna in columnas_competencia):
    df_inferencia_transformado['precio_Competencia'] = df_inferencia_transformado[columnas_competencia].mean(axis=1)
    df_inferencia_transformado['ratio_precio'] = df_inferencia_transformado['precio_venta'] / df_inferencia_transformado['precio_Competencia']
    df_inferencia_transformado = df_inferencia_transformado.drop(columns=columnas_competencia)

df_inferencia_transformado[['fecha', 'producto_id', 'precio_venta', 'precio_Competencia', 'ratio_precio', 'descuento_porcentaje']].head()

,fecha,producto_id,precio_venta,precio_Competencia,ratio_precio,descuento_porcentaje
0,2025-10-25,PROD_001,113.13,102.573333,1.102918,-1.626087
24,2025-10-26,PROD_001,105.75,98.356667,1.075169,-8.043478
48,2025-10-27,PROD_001,114.95,97.740000,1.176079,-0.043478
72,2025-10-28,PROD_001,117.31,103.146667,1.137313,2.008696
96,2025-10-29,PROD_001,108.10,100.520000,1.075408,-6.000000


## Codificación one-hot de variables categóricas

Se codifican `nombre`, `categoria` y `subcategoria` con el mismo prefijo `_h` utilizado en el entrenamiento.

In [9]:
# La celda es idempotente: si ya se ejecutó antes, las columnas dummy
# 'nombre_h_...' ya existen y no se vuelven a generar (evita duplicados).
ya_codificado = any(columna.startswith('nombre_h_') for columna in df_inferencia_transformado.columns)

if not ya_codificado:
    df_inferencia_transformado['nombre_h'] = df_inferencia_transformado['nombre']
    df_inferencia_transformado['categoria_h'] = df_inferencia_transformado['categoria']
    df_inferencia_transformado['subcategoria_h'] = df_inferencia_transformado['subcategoria']

    columnas_one_hot = ['nombre_h', 'categoria_h', 'subcategoria_h']
    df_inferencia_transformado = pd.get_dummies(
        df_inferencia_transformado,
        columns=columnas_one_hot,
        prefix=columnas_one_hot,
        dtype=int
    )

df_inferencia_transformado.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,0,0,0,0,0,0,0,0,1,0
24,2025-10-26,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,20.0,105.75,2115.00,...,0,0,0,0,0,0,0,0,1,0
48,2025-10-27,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,16.0,114.95,1839.20,...,0,0,0,0,0,0,0,0,1,0
72,2025-10-28,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,15.0,117.31,1759.65,...,0,0,0,0,0,0,0,0,1,0
96,2025-10-29,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,14.0,108.10,1513.40,...,0,0,0,0,0,0,0,0,1,0


## Alineación de columnas con `df_integrado`

Se añaden con valor 0 las columnas one-hot que no aparecen en `df_inferencia` (por ejemplo, categorías sin observaciones en el periodo de inferencia) y se eliminan columnas que no formen parte de la estructura de entrenamiento, para dejar exactamente las mismas 97 columnas y en el mismo orden.

In [10]:
columnas_faltantes = [columna for columna in columnas_objetivo if columna not in df_inferencia_transformado.columns]
for columna in columnas_faltantes:
    df_inferencia_transformado[columna] = 0

columnas_extra = [columna for columna in df_inferencia_transformado.columns if columna not in columnas_objetivo]
if columnas_extra:
    df_inferencia_transformado = df_inferencia_transformado.drop(columns=columnas_extra)

df_inferencia_transformado = df_inferencia_transformado[columnas_objetivo].copy()

print(f'Columnas añadidas (no presentes en inferencia): {columnas_faltantes}')
print(f'Columnas eliminadas (no presentes en df_integrado): {columnas_extra}')
print(f'Dimensiones finales tras alinear columnas: {df_inferencia_transformado.shape}')

Columnas añadidas (no presentes en inferencia): []
Columnas eliminadas (no presentes en df_integrado): []
Dimensiones finales tras alinear columnas: (888, 97)


## Eliminación de octubre y guardado del resultado

Una vez terminado **todo** el preprocesamiento (lags, media móvil, precios, one-hot y alineación de columnas), se eliminan únicamente los registros de octubre (`mes == 10`), dejando **intactos** todos los registros de noviembre (aunque algunas de sus columnas de lag/media móvil contengan NaN, ya que son las que el modelo debe predecir). El resultado se guarda en `data/processed/df_inferencia_transformado.csv`.

In [11]:
df_inferencia_transformado = df_inferencia_transformado.loc[df_inferencia_transformado['mes'] != 10].copy()

ruta_salida = '../data/processed/df_inferencia_transformado.csv'
df_inferencia_transformado.to_csv(ruta_salida, index=False)

print(f'df_inferencia_transformado guardado en: {ruta_salida}')
print(f'Filas guardadas (octubre eliminado, noviembre intacto): {len(df_inferencia_transformado)}')
print(f'Columnas finales: {len(df_inferencia_transformado.columns)}')
# Se muestra un registro por producto (en vez de head(), que solo enseñaria
# PROD_001 repetido por el orden de sort_values) para confirmar que los 24
# productos quedaron presentes tras el preprocesamiento.
df_inferencia_transformado.groupby('producto_id').head(1)[['fecha', 'producto_id', 'nombre', 'mes']]

df_inferencia_transformado guardado en: ../data/processed/df_inferencia_transformado.csv
Filas guardadas (octubre eliminado, noviembre intacto): 720
Columnas finales: 97


,fecha,producto_id,nombre,mes
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,11
169,2025-11-01,PROD_002,Adidas Ultraboost 23,11
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,11
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,11
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,11
173,2025-11-01,PROD_006,Adidas Own The Run Jacket,11
174,2025-11-01,PROD_007,Puma Velocity Nitro 2,11
175,2025-11-01,PROD_008,Reebok Floatride Energy 5,11
176,2025-11-01,PROD_009,Bowflex SelectTech 552,11
177,2025-11-01,PROD_010,Domyos BM900,11


## Carga del modelo de forecasting

Se carga el modelo entrenado (`models/modelo_final.joblib`) y se reconstruyen las `columnas_predictoras` siguiendo exactamente la misma lógica usada en `entrenamiento.ipynb`: se excluyen `fecha`, `ingresos`, `unidades_vendidas` (la variable objetivo) y cualquier columna de tipo texto (`object`).

In [12]:
import joblib

ruta_modelo = '../models/modelo_final.joblib'
modelo_forecasting = joblib.load(ruta_modelo)

columna_objetivo = 'unidades_vendidas'
columnas_excluidas = ['fecha', 'ingresos', columna_objetivo]
columnas_object = df_inferencia_transformado.select_dtypes(include='object').columns.tolist()
columnas_excluidas = columnas_excluidas + [
    columna for columna in columnas_object if columna not in columnas_excluidas
]
columnas_predictoras = [
    columna for columna in df_inferencia_transformado.columns if columna not in columnas_excluidas
]

print(f'Modelo cargado desde: {ruta_modelo}')
print(f'Columnas predictoras utilizadas ({len(columnas_predictoras)}): {columnas_predictoras}')

Modelo cargado desde: ../models/modelo_final.joblib
Columnas predictoras utilizadas (87): ['precio_base', 'es_estrella', 'precio_venta', 'anio', 'mes', 'trimestre', 'dia_mes', 'dia_anio', 'dia_semana', 'semana_iso', 'semana_mes', 'es_fin_semana', 'es_inicio_mes', 'es_fin_mes', 'es_inicio_trimestre', 'es_fin_trimestre', 'es_inicio_anio', 'es_fin_anio', 'es_quincena', 'es_festivo', 'es_black_friday', 'es_cyber_monday', 'dias_desde_black_friday', 'dias_hasta_black_friday', 'dias_desde_inicio_datos', 'dias_hasta_fin_mes', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos', 'dia_anio_sin', 'dia_anio_cos', 'lag_1_unidades_vendidas', 'lag_2_unidades_vendidas', 'lag_3_unidades_vendidas', 'lag_4_unidades_vendidas', 'lag_5_unidades_vendidas', 'lag_6_unidades_vendidas', 'lag_7_unidades_vendidas', 'media_movil_7_dias', 'descuento_porcentaje', 'precio_Competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_

C:\Users\rodri\AppData\Local\Temp\ipykernel_21884\3680913835.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_object = df_inferencia_transformado.select_dtypes(include='object').columns.tolist()


## Generación de predicciones de forecasting

Se aplica el modelo sobre `df_inferencia_transformado` y se guarda el resultado en una nueva columna `unidades_vendidas_predichas`.

In [13]:
X_inferencia = df_inferencia_transformado[columnas_predictoras]
df_inferencia_transformado['unidades_vendidas_predichas'] = modelo_forecasting.predict(X_inferencia)

print(f'Predicciones generadas: {len(df_inferencia_transformado)}')
print(f'Productos distintos con predicción: {df_inferencia_transformado["producto_id"].nunique()}')

# Un registro por producto para verificar que todos los productos tienen predicción.
df_inferencia_transformado.groupby('producto_id').head(1)[
    ['fecha', 'producto_id', 'nombre', 'unidades_vendidas_predichas']
]

Predicciones generadas: 720
Productos distintos con predicción: 24


,fecha,producto_id,nombre,unidades_vendidas_predichas
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,13.869489
169,2025-11-01,PROD_002,Adidas Ultraboost 23,12.348656
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,4.532686
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,3.877314
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,4.038641
173,2025-11-01,PROD_006,Adidas Own The Run Jacket,2.458151
174,2025-11-01,PROD_007,Puma Velocity Nitro 2,4.516806
175,2025-11-01,PROD_008,Reebok Floatride Energy 5,4.116703
176,2025-11-01,PROD_009,Bowflex SelectTech 552,2.977293
177,2025-11-01,PROD_010,Domyos BM900,3.171191


## Guardado final con las predicciones

Se vuelve a guardar `data/processed/df_inferencia_transformado.csv`, ahora incluyendo la columna `unidades_vendidas_predichas` generada por el modelo.

In [14]:
ruta_salida = '../data/processed/df_inferencia_transformado.csv'
df_inferencia_transformado.to_csv(ruta_salida, index=False)

print(f'df_inferencia_transformado guardado en: {ruta_salida}')
print(f'Filas guardadas: {len(df_inferencia_transformado)}')
print(f'Columnas finales (incluyendo la predicción): {len(df_inferencia_transformado.columns)}')

# Muestra de varios productos distintos (no solo el primero) para confirmar
# visualmente que hay predicciones para los 24 productos.
df_inferencia_transformado[['fecha', 'producto_id', 'nombre', 'unidades_vendidas_predichas']].sample(10, random_state=42)

df_inferencia_transformado guardado en: ../data/processed/df_inferencia_transformado.csv
Filas guardadas: 720
Columnas finales (incluyendo la predicción): 98


,fecha,producto_id,nombre,unidades_vendidas_predichas
419,2025-11-11,PROD_012,Domyos Kit Mancuernas 20kg,2.506182
657,2025-11-21,PROD_010,Domyos BM900,2.524442
745,2025-11-25,PROD_002,Adidas Ultraboost 23,2.716117
606,2025-11-19,PROD_007,Puma Velocity Nitro 2,2.424372
255,2025-11-04,PROD_016,Trek Marlin 7,2.574075
290,2025-11-06,PROD_003,Asics Gel Nimbus 25,2.801746
242,2025-11-04,PROD_003,Asics Gel Nimbus 25,4.781903
772,2025-11-26,PROD_005,Nike Dri-FIT Miler,2.814991
705,2025-11-23,PROD_010,Domyos BM900,2.554067
181,2025-11-01,PROD_014,Sveltus Kettlebell 12kg,4.922893


In [15]:
df_inferencia_transformado.columns


Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'anio', 'mes', 'trimestre', 'dia_mes', 'dia_anio',
       'dia_semana', 'nombre_dia', 'nombre_mes', 'semana_iso', 'semana_mes',
       'es_fin_semana', 'es_inicio_mes', 'es_fin_mes', 'es_inicio_trimestre',
       'es_fin_trimestre', 'es_inicio_anio', 'es_fin_anio', 'es_quincena',
       'es_festivo', 'nombre_festivo', 'es_black_friday', 'es_cyber_monday',
       'dias_desde_black_friday', 'dias_hasta_black_friday',
       'dias_desde_inicio_datos', 'dias_hasta_fin_mes', 'dia_semana_sin',
       'dia_semana_cos', 'mes_sin', 'mes_cos', 'dia_anio_sin', 'dia_anio_cos',
       'lag_1_unidades_vendidas', 'lag_2_unidades_vendidas',
       'lag_3_unidades_vendidas', 'lag_4_unidades_vendidas',
       'lag_5_unidades_vendidas', 'lag_6_unidades_vendidas',
       'lag_7_unidades_vendidas', 'media_movil_7_dias', 'descuento_porcentaje',
  